# Video Transformer (Phase 6)

Runs in Google Colab. Actual training happens here, never on a laptop.
This notebook calls `evat.models.transformer.*` — it does not reimplement
attention/positional-encoding/block logic.

The temporal Transformer is implemented from scratch using PyTorch
primitives; no complete pretrained Video Transformer architecture is
used. The only pretrained component anywhere in this pipeline is Phase
5's spatial CNN feature encoder (MobileNetV3-Small).

In [ ]:
%pip install -q -e .

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
device = "cuda" if torch.cuda.is_available() else "cpu"

## SMOKE RUN (synthetic data)

Verify the model builds, tensors move to the GPU correctly, forward and
backward pass work, and checkpointing works — with synthetic tensors —
before touching any real feature cache.

In [ ]:
from evat.models.transformer.config import TransformerConfig
from evat.models.transformer.model import VideoTransformer

config = TransformerConfig.from_yaml("configs/transformer.yaml")
model = VideoTransformer(config).to(device)
print("parameters:", model.num_parameters())

batch_size, seq_len = 4, 16
features = torch.randn(batch_size, seq_len, config.feature_dim, device=device)
validity_mask = torch.ones(batch_size, seq_len, dtype=torch.bool, device=device)
target = torch.randint(0, config.num_classes, (batch_size,), device=device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
output = model(features, validity_mask=validity_mask)
loss = torch.nn.functional.cross_entropy(output.logits, target)
loss.backward()
optimizer.step()
print("smoke loss:", loss.item())

torch.save({"model_state": model.state_dict(), "config": config}, "/content/smoke_checkpoint.pt")
print("checkpoint saved/loaded OK:", torch.load("/content/smoke_checkpoint.pt") is not None)

## SMALL TRAINING RUN (real features, if the Phase 5 feature cache exists)

Only run this if `results/features/cache/` (or wherever `configs/features.yaml`
points) has already been populated by Phase 5's notebook. This is a small
run on a subset, not full-dataset training.

In [ ]:
from pathlib import Path

cache_dir = Path("results/features/cache")
if not cache_dir.exists() or not any(cache_dir.glob("*.npy")):
    print("No Phase 5 feature cache found — run notebooks/05_feature_extraction.ipynb first.")
else:
    # Load cached TemporalFeatureSequences here (built the same way Phase 5's
    # notebook constructs them via evat.features.temporal), assemble a small
    # supervised subset, and run a short training loop with VideoTransformer
    # exactly as in the smoke run above, but on real feature sequences.
    # Left as a TODO until an actual downstream label source is wired in —
    # Phase 6 has no anomaly/classification labels of its own yet.
    print("Real-feature training requires a labeled downstream task, not yet defined.")

## Record configuration and results

Only fill in with values actually printed by cells above.

In [ ]:
import subprocess

git_commit = subprocess.check_output(["git", "rev-parse", "HEAD"]).decode().strip()
print("git_commit:", git_commit)
print("config:", config)
print("device:", device)
print("parameter_count:", model.num_parameters())

Update `docs/experiments.md` with the actual printed values above. Do not
hand-edit numbers not produced by this notebook.